# Understanding Logistic Regression

This notebook teaches **logistic regression** using NBA player game data from **S25.csv**. You'll learn when to use it, its assumptions, pros and cons, failure cases, and see it in action on real data.

## 1. When to Use Logistic Regression

Use **logistic regression** when:

- **Target is binary (or categorical)** — You're predicting a class: win/loss, yes/no, default/repay, etc.
- **You want probabilities** — The model outputs P(Y=1), which you can threshold (e.g. at 0.5) for classification.
- **Interpretability matters** — Coefficients relate to **log-odds**; you can interpret "one more unit of X multiplies odds by exp(β)."
- **Linear decision boundary in the predictors** — Log-odds are linear in X: $$\log\frac{p}{1-p} = \beta_0 + \beta_1 X_1 + \cdots + \beta_k X_k$$
- **Baseline classifier** — Simple, fast, and robust for many binary classification problems.

**Example with our data:** Predicting **WL** (win/loss) from a player's stats in that game (e.g. PTS, PLUS_MINUS, AST) — did the team win when this player had this performance?

## 2. Assumptions of Logistic Regression

| Assumption | Meaning | Notes |
|------------|--------|-------|
| **Binary/categorical outcome** | Y is 0/1 (or one-hot for multinomial). | Logistic regression is for classification. |
| **Linearity of log-odds** | Log-odds are linear in the predictors: $$\log\frac{p}{1-p} = \beta_0 + \beta_1 X_1 + \cdots + \beta_k X_k$$ | Check with residual-type diagnostics or by adding interaction terms. |
| **Independence** | Observations are independent (e.g. no repeated games/players without accounting for it). | Use clustered SE or mixed-effects models if you have repeated measures. |
| **No perfect separation** | No linear combination of X perfectly separates the two classes. | Otherwise coefficients blow up; use regularization (e.g. Ridge/Lasso). |
| **Large enough sample** | Rule of thumb: ~10–15 events per predictor. | Few events and many features → overfitting. |

Unlike linear regression, we do **not** assume normality or homoscedasticity of the outcome; the model is built for binary Y.

## 3. Pros and Cons

### Pros
- **Designed for binary (or multi-class) outcomes** — Models probability directly; no need to fake a continuous target.
- **Interpretable** — Coefficients = change in log-odds; exp(β) = odds ratio per unit change in X.
- **Probabilistic output** — Get P(Y=1) for ranking, thresholding, or downstream decisions.
- **No normality assumption** — Works well for 0/1 data.
- **Fast and stable** — Convex optimization; widely available in every stats/ML library.

### Cons
- **Assumes linearity in log-odds** — Nonlinear relationships need transforms or interaction terms.
- **Can be unstable with perfect or near-perfect separation** — Coefficients can explode; use regularization.
- **Not for continuous targets** — Use linear regression (or other regression) for numeric Y.
- **Needs enough events** — With many predictors and few positive (or negative) cases, estimates are unreliable.

## 4. Failure Cases

Logistic regression can **fail or mislead** when:

1. **Perfect separation** — A linear combination of X perfectly predicts Y; coefficients → ±∞. **Fix:** Regularization (e.g. C small in sklearn) or remove/fewer features.
2. **Nonlinearity in log-odds** — True relationship is curved or has strong interactions; model underfits. **Fix:** Add polynomial or interaction terms, or use a more flexible model.
3. **Severe class imbalance** — Very few positives (or negatives); model may predict the majority class always. **Fix:** Class weights, oversampling, or threshold tuning.
4. **Too many features / multicollinearity** — Unstable coefficients, overfitting. **Fix:** Regularization, feature selection, or PCA.
5. **Wrong link** — For count or continuous outcomes, use Poisson or linear regression instead.
6. **Non-independence** — Repeated observations (e.g. same player many games); standard errors too small. **Fix:** Clustered or robust SE, or mixed-effects model.

---
## 5. Load and Prepare S25 Data

We'll predict **WL** (win = 1, loss = 0) from player performance: **PTS**, **PLUS_MINUS**, **AST**, **MIN**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

df = pd.read_csv('../S25.csv')
# Binary target: W=1, L=0
df['WIN'] = (df['WL'] == 'W').astype(int)
target = 'WIN'
features = ['PTS', 'PLUS_MINUS', 'AST', 'MIN']
cols = [target] + features
sub = df[cols].dropna()
print("Shape:", sub.shape)
print("Win rate:", sub[target].mean())
print(sub.describe())

### Distribution of target and one predictor

PLUS_MINUS (point differential when player was on court) should be higher in wins.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sub['WIN'].value_counts().sort_index().plot(kind='bar', ax=ax[0])
ax[0].set_title('Win (1) vs Loss (0)')
ax[0].set_xticklabels(['Loss', 'Win'])
sns.boxplot(data=sub, x='WIN', y='PLUS_MINUS', ax=ax[1])
ax[1].set_title('PLUS_MINUS by outcome')
plt.tight_layout()
plt.show()

---
## 6. Fit Logistic Regression

Model: **log(odds of win) = β₀ + β₁·PTS + β₂·PLUS_MINUS + β₃·AST + β₄·MIN**

In [ ]:
X = sub[features]
y = sub[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Coefficients (log-odds per 1-unit change in feature):")
for name, coef in zip(features, model.coef_[0]):
    print(f"  {name}: {coef:.3f}  →  odds ratio: {np.exp(coef):.3f}")
print(f"  Intercept: {model.intercept_[0]:.3f}")
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\n", classification_report(y_test, y_pred, target_names=['Loss', 'Win']))

### ROC curve

Trade-off between true positive rate and false positive rate as we vary the classification threshold.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f'Logistic (AUC = {roc_auc_score(y_test, y_proba):.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()

---
## 7. Failure Case: Class Imbalance

If one class is rare, the model may predict the majority class almost always. We simulate this by training on a heavily downsampled "Win" class and show how accuracy can be misleading.

In [ ]:
# Heavily imbalanced sample: keep most losses, few wins
losses = sub[sub['WIN'] == 0]
wins_df = sub[sub['WIN'] == 1]
wins = wins_df.sample(n=min(200, len(wins_df)), random_state=42)
imbalanced = pd.concat([losses, wins], ignore_index=True).sample(frac=1, random_state=42)
Xi = imbalanced[features]
yi = imbalanced[target]
Xi_train, Xi_test, yi_train, yi_test = train_test_split(Xi, yi, test_size=0.2, random_state=42)

model_imb = LogisticRegression(max_iter=1000, random_state=42)
model_imb.fit(Xi_train, yi_train)
yi_pred = model_imb.predict(Xi_test)
print("Imbalanced train set - Win rate:", yi_train.mean())
print("Accuracy (can be misleading):", accuracy_score(yi_test, yi_pred))
print("Confusion matrix:\n", confusion_matrix(yi_test, yi_pred))
print("\n→ With imbalance, always check precision/recall and ROC-AUC, not just accuracy.")

---
## 8. Summary

| Topic | Takeaway |
|-------|----------|
| **When to use** | Binary (or multi-class) target; you want probabilities and interpretable coefficients. |
| **Assumptions** | Linearity of log-odds, independence, no perfect separation, enough events per predictor. |
| **Pros** | Built for classification, interpretable (odds ratios), probabilistic output, stable. |
| **Cons** | Linear in log-odds, can fail with perfect separation or severe imbalance. |
| **Failure cases** | Perfect separation, nonlinearity, class imbalance, too many features, non-independence. |

On S25 data, logistic regression predicts **win/loss** from **PTS**, **PLUS_MINUS**, **AST**, and **MIN**. Use probability outputs and ROC-AUC when classes are imbalanced.